# Working with LLM APIs

In this notebook,  we will review how to use the OpenAI API Python SDK to send prompts to the following APIs:
- OpenAI


General requirements:
- APIs keys
- Python SDKs for the APIs

Let's start by import the os library:

In [5]:
from openai import OpenAI
import os

In [7]:
openai_api_key = os.environ.get("OPENAI_API_KEY")
#print(openai_api_key)

We than define the following prompt:

In [3]:
content_prompt = """
I am working with a dataset that contains information about the national bridge inventory.
I want to create a SQL query that calculates the total number of bridges by state.
"""

Next, we define the number of tokens and temperature:

In [4]:
temperature = 0
max_tokens = 5000

# The OpenAI API

In this demo, we will review two models from the OpenAI API:
- [GPT4.1](https://platform.openai.com/docs/models/gpt-4.1)
- [GPT5](https://platform.openai.com/docs/models/gpt-5)

We will use the `chat.completion` method to send a GET request to the OpenAI API. The method arguments may change over time and differ between different models or be deprecated. Therefore, please check the method's [documentation](https://platform.openai.com/docs/api-reference/chat) before using it.

We will start by loading the API key and set the client:

In [5]:
client = OpenAI(api_key= openai_api_key)

> Note, the function has the OpenAI base URL as default, and therefore, we don't need to set it up.

### GPT4.1

In the first example, we will use GPT4.1 model. We will use the `message` argument to set the prompt, the `temperature` and `max_completion_tokens` arguments to define he corresponding arguments. Last but not least, we define the model as `gpt-4.1`:

In [6]:
response_gpt4_1 = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", "content": content_prompt}],
    max_completion_tokens=max_tokens,
    temperature=temperature,
)

Let's review the output:

In [7]:
print(response_gpt4_1)

ChatCompletion(id='chatcmpl-DHZ8LNolItYCCJv4sYwEvzzlvYQXh', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Certainly! Assuming your table is named `bridges` and there is a column called `state` that contains the state abbreviation or name, you can use the following SQL query to calculate the total number of bridges by state:\n\n```sql\nSELECT state, COUNT(*) AS total_bridges\nFROM bridges\nGROUP BY state\nORDER BY total_bridges DESC;\n```\n\n**Explanation:**\n- `SELECT state, COUNT(*) AS total_bridges`: Selects the state and counts the number of rows (bridges) for each state.\n- `FROM bridges`: Specifies the table.\n- `GROUP BY state`: Groups the results by the state column.\n- `ORDER BY total_bridges DESC`: Orders the results by the total number of bridges in descending order (most bridges first).\n\nIf your column or table names are different, adjust them accordingly. Let me know if you need help with a specific schema!', refusal=

We will extract the response:

In [8]:
print(response_gpt4_1.choices[0].message.content)

Certainly! Assuming your table is named `bridges` and there is a column called `state` that contains the state abbreviation or name, you can use the following SQL query to calculate the total number of bridges by state:

```sql
SELECT state, COUNT(*) AS total_bridges
FROM bridges
GROUP BY state
ORDER BY total_bridges DESC;
```

**Explanation:**
- `SELECT state, COUNT(*) AS total_bridges`: Selects the state and counts the number of rows (bridges) for each state.
- `FROM bridges`: Specifies the table.
- `GROUP BY state`: Groups the results by the state column.
- `ORDER BY total_bridges DESC`: Orders the results by the total number of bridges in descending order (most bridges first).

If your column or table names are different, adjust them accordingly. Let me know if you need help with a specific schema!


### GPT5

Next, we will use the GTP5 model. Note that this models is a bit different from previous OpenAI models. It uses a different sampling method; therefore, the `temperature` is invalid. Instead, if you wish the model to return a deterministic response, you should use the `seed` argument.

In [9]:
response_gpt5 = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "user", "content": content_prompt}],
    max_completion_tokens = max_tokens
)

Let's print the model response:

In [10]:
print(response_gpt5.choices[0].message.content)

Here are a few options depending on how your dataset stores the state field.

1) If your table has a state name or abbreviation column (e.g., state):
SELECT state, COUNT(*) AS bridge_count
FROM nbi_bridges
GROUP BY state
ORDER BY bridge_count DESC;

2) If your table uses the NBI/FIPS state code (e.g., STATE_CODE_001) and you don’t have a lookup table:
SELECT STATE_CODE_001 AS state_fips, COUNT(*) AS bridge_count
FROM nbi_bridges
GROUP BY STATE_CODE_001
ORDER BY bridge_count DESC;

3) If you have a states lookup table (recommended), e.g., us_states(state_fips, state_abbr, state_name):
SELECT s.state_abbr, COUNT(*) AS bridge_count
FROM nbi_bridges b
JOIN us_states s
  ON LPAD(CAST(b.STATE_CODE_001 AS CHAR(2)), 2, '0') = s.state_fips
GROUP BY s.state_abbr
ORDER BY bridge_count DESC;

Tip: If your dataset contains multiple reporting years, add a WHERE clause to select a specific year (e.g., WHERE data_year = 2023) so you don’t double-count bridges across years.

If you share your exact tab

# Resources

- OpenAI API [documentation](https://platform.openai.com/docs/api-reference/introduction)
- Gemini API [documentation](https://ai.google.dev/gemini-api/docs)
- Claude API [documentation](https://docs.anthropic.com/en/home)

# Optimize SQL Prompts: Context Engineering for Better Query Results

Previously, we tested the below prompt:

>I am working with a dataset that contains information about the Red30 Tech company's product sales online. 
>I want to create a SQL query that calculates the total sales by state.

While it returns a SQL code, it was generic and cannot be used out of the box as it missing critical context. 

In this example we will see how we can improve the LLM's response with providing additional context.

Next, let's add to the original prompt additional information with the table and columns names:

In [8]:
client = OpenAI(api_key= openai_api_key)

In [9]:
content_prompt = """
I am working with a dataset that contains information about the national bridge inventory data for Alaska. 

I want to create a SQL query that calculates the total number of bridges in Alaska.

The table name is nbi_2025_ak which stands for "National Bridge Inventory 2025 Alaska", 
all the column headers are listed herein:
STATE_CODE_001
STRUCTURE_NUMBER_008
RECORD_TYPE_005A
ROUTE_PREFIX_005B
SERVICE_LEVEL_005C
ROUTE_NUMBER_005D
DIRECTION_005E
HIGHWAY_DISTRICT_002
COUNTY_CODE_003
PLACE_CODE_004
FEATURES_DESC_006A
CRITICAL_FACILITY_006B
FACILITY_CARRIED_007
LOCATION_009
MIN_VERT_CLR_010
KILOPOINT_011
BASE_HWY_NETWORK_012
LRS_INV_ROUTE_013A
SUBROUTE_NO_013B
LAT_016
LONG_017
DETOUR_KILOS_019
TOLL_020
MAINTENANCE_021
OWNER_022
FUNCTIONAL_CLASS_026
YEAR_BUILT_027
TRAFFIC_LANES_ON_028A
TRAFFIC_LANES_UND_028B
ADT_029
YEAR_ADT_030
DESIGN_LOAD_031
APPR_WIDTH_MT_032
MEDIAN_CODE_033
DEGREES_SKEW_034
STRUCTURE_FLARED_035
RAILINGS_036A
TRANSITIONS_036B
APPR_RAIL_036C
APPR_RAIL_END_036D
HISTORY_037
NAVIGATION_038
NAV_VERT_CLR_MT_039
NAV_HORR_CLR_MT_040
OPEN_CLOSED_POSTED_041
SERVICE_ON_042A
SERVICE_UND_042B
STRUCTURE_KIND_043A
STRUCTURE_TYPE_043B
APPR_KIND_044A
APPR_TYPE_044B
MAIN_UNIT_SPANS_045
APPR_SPANS_046
HORR_CLR_MT_047
MAX_SPAN_LEN_MT_048
STRUCTURE_LEN_MT_049
LEFT_CURB_MT_050A
RIGHT_CURB_MT_050B
ROADWAY_WIDTH_MT_051
DECK_WIDTH_MT_052
VERT_CLR_OVER_MT_053
VERT_CLR_UND_REF_054A
VERT_CLR_UND_054B
LAT_UND_REF_055A
LAT_UND_MT_055B
LEFT_LAT_UND_MT_056
DECK_COND_058
SUPERSTRUCTURE_COND_059
SUBSTRUCTURE_COND_060
CHANNEL_COND_061
CULVERT_COND_062
OPR_RATING_METH_063
OPERATING_RATING_064
INV_RATING_METH_065
INVENTORY_RATING_066
STRUCTURAL_EVAL_067
DECK_GEOMETRY_EVAL_068
UNDCLRENCE_EVAL_069
POSTING_EVAL_070
WATERWAY_EVAL_071
APPR_ROAD_EVAL_072
WORK_PROPOSED_075A
WORK_DONE_BY_075B
IMP_LEN_MT_076
DATE_OF_INSPECT_090
INSPECT_FREQ_MONTHS_091
FRACTURE_092A
UNDWATER_LOOK_SEE_092B
SPEC_INSPECT_092C
FRACTURE_LAST_DATE_093A
UNDWATER_LAST_DATE_093B
SPEC_LAST_DATE_093C
BRIDGE_IMP_COST_094
ROADWAY_IMP_COST_095
TOTAL_IMP_COST_096
YEAR_OF_IMP_097
OTHER_STATE_CODE_098A
OTHER_STATE_PCNT_098B
OTHR_STATE_STRUC_NO_099
STRAHNET_HIGHWAY_100
PARALLEL_STRUCTURE_101
TRAFFIC_DIRECTION_102
TEMP_STRUCTURE_103
HIGHWAY_SYSTEM_104
FEDERAL_LANDS_105
YEAR_RECONSTRUCTED_106
DECK_STRUCTURE_TYPE_107
SURFACE_TYPE_108A
MEMBRANE_TYPE_108B
DECK_PROTECTION_108C
PERCENT_ADT_TRUCK_109
NATIONAL_NETWORK_110
PIER_PROTECTION_111
BRIDGE_LEN_IND_112
SCOUR_CRITICAL_113
FUTURE_ADT_114
YEAR_OF_FUTURE_ADT_115
MIN_NAV_CLR_MT_116
FED_AGENCY
SUBMITTED_BY
BRIDGE_CONDITION
LOWEST_RATING
DECK_AREA
"""

Let's now use the GPT5 model to send the prompt:

In [10]:
response = client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "user", "content": content_prompt}],
    max_completion_tokens=max_tokens,
)

print(response.choices[0].message.content)

NameError: name 'max_tokens' is not defined